# 6. Threshold 编码脉冲

分箱取最大值 + 阈值化。

In [ ]:
# 修改这两个参数即可生成不同时间步和目标发放率的数据集。
T = 200
TARGET_SPIKE_RATE = 0.10

# 仅当明确需要重新生成同一参数目录时改为 True。
OVERWRITE = False

In [ ]:
import hashlib
import json
import math
import os
import re
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import torch
from tqdm.auto import tqdm


def find_project_root():
    # 同时兼容从项目根目录或 notebook 所在目录启动 Jupyter。
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'src' / 'data_prepare').is_dir() and (candidate / 'CapgMyo_data').is_dir():
            return candidate
    raise RuntimeError('找不到同时包含 src/data_prepare 和 CapgMyo_data 的项目根目录。')


def format_rate(rate):
    return f'{rate:.12g}'


if isinstance(T, bool) or not isinstance(T, int) or not 2 <= T <= 1000:
    raise ValueError(f'T 必须是 [2, 1000] 范围内的整数，当前为 {T!r}。')
if isinstance(TARGET_SPIKE_RATE, bool) or not isinstance(TARGET_SPIKE_RATE, (int, float)):
    raise TypeError('TARGET_SPIKE_RATE 必须是数值。')
if not math.isfinite(float(TARGET_SPIKE_RATE)) or not 0 < TARGET_SPIKE_RATE < 1:
    raise ValueError('TARGET_SPIKE_RATE 必须位于 (0, 1) 区间。')

PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / 'CapgMyo_data'
OUTPUT_ROOT = DATA_ROOT / 'threshold_encoding_spike' / f'T_{T}_target_rate{format_rate(TARGET_SPIKE_RATE)}'
OUTPUT_MANIFEST_PATH = OUTPUT_ROOT / 'manifest.json'
SPLIT_NAMES = ('train', 'val', 'test')
EXPECTED_COUNTS = {'train': 1008, 'val': 144, 'test': 288}
SPLIT_REPETITIONS = {
    'train': {1, 3, 4, 5, 6, 7, 8},
    'val': {10},
    'test': {2, 9},
}
SOURCE_CONFIGS = {
    'raw_polarity': {
        'root': DATA_ROOT / 'raw_polarity',
        'manifest': DATA_ROOT / 'raw_polarity' / 'manifest.json',
        'channels': 2,
        'nonnegative': True,
        'score_transform': 'identity',
    },
}
METADATA_KEYS = ('label', 'subject_id', 'gesture_id', 'repetition_id')
FILE_NAME_PATTERN = re.compile(r's(\d+)_g(\d+)_r(\d+)\.pt$', re.IGNORECASE)
HISTOGRAM_BINS = 65536
NATIVE_TIME_STEPS = 1000
TIME_BIN_INDEX = torch.div(
    torch.arange(NATIVE_TIME_STEPS, dtype=torch.long) * T,
    NATIVE_TIME_STEPS,
    rounding_mode='floor',
)

print(f'项目根目录：{PROJECT_ROOT}')
print(f'输出目录：{OUTPUT_ROOT}')
print(f'T={T}, TARGET_SPIKE_RATE={TARGET_SPIKE_RATE:.2%}, OVERWRITE={OVERWRITE}')

In [ ]:
def load_pt(path):
    try:
        return torch.load(path, map_location='cpu', weights_only=True)
    except TypeError:
        # 兼容尚未提供 weights_only 参数的旧版 PyTorch。
        return torch.load(path, map_location='cpu')


def write_json_atomic(path, value):
    temporary_path = path.with_suffix(path.suffix + '.tmp')
    temporary_path.write_text(
        json.dumps(value, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )
    os.replace(temporary_path, path)


def save_pt_atomic(path, payload):
    # 只在完整写入后替换目标，避免中断留下可见但不完整的样本。
    temporary_path = path.with_suffix(path.suffix + '.tmp')
    torch.save(payload, temporary_path)
    os.replace(temporary_path, path)


def sha256sum(path):
    digest = hashlib.sha256()
    with path.open('rb') as file_handle:
        for chunk in iter(lambda: file_handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def scalar_int(value, key, path):
    if not isinstance(value, torch.Tensor) or value.ndim != 0 or value.dtype != torch.int64:
        raise TypeError(f'{path} 中 {key} 必须是 torch.int64 标量。')
    return int(value)


def validate_metadata(payload, path):
    missing = {'data', *METADATA_KEYS} - set(payload)
    if missing:
        raise KeyError(f'{path} 缺少字段：{sorted(missing)}')

    metadata = {key: scalar_int(payload[key], key, path) for key in METADATA_KEYS}
    if metadata['label'] != metadata['gesture_id'] - 1 or not 0 <= metadata['label'] <= 7:
        raise ValueError(f'{path} 中 label/gesture_id 不合法。')
    if not 1 <= metadata['subject_id'] <= 18:
        raise ValueError(f'{path} 中 subject_id 越界。')
    if not 1 <= metadata['gesture_id'] <= 8:
        raise ValueError(f'{path} 中 gesture_id 越界。')
    if not 1 <= metadata['repetition_id'] <= 10:
        raise ValueError(f'{path} 中 repetition_id 越界。')

    match = FILE_NAME_PATTERN.fullmatch(path.name)
    if match is None:
        raise ValueError(f'文件名不符合 sXX_gXX_rXX.pt：{path.name}')
    file_metadata = tuple(map(int, match.groups()))
    payload_metadata = (
        metadata['subject_id'],
        metadata['gesture_id'],
        metadata['repetition_id'],
    )
    if file_metadata != payload_metadata:
        raise ValueError(f'{path} 的文件名与内部元数据不一致。')
    return metadata


def validate_source_payload(payload, path, config):
    if not isinstance(payload, dict):
        raise TypeError(f'{path} 的内容必须是字典。')
    metadata = validate_metadata(payload, path)
    data = payload['data']
    expected_shape = (NATIVE_TIME_STEPS, config['channels'], 8, 16)
    if not isinstance(data, torch.Tensor):
        raise TypeError(f'{path} 中 data 必须是 Tensor。')
    if tuple(data.shape) != expected_shape or data.dtype != torch.float32:
        raise ValueError(
            f'{path} 中 data 为 shape={tuple(data.shape)}, dtype={data.dtype}；'
            f'预期 shape={expected_shape}, dtype=torch.float32。'
        )
    if not torch.isfinite(data).all():
        raise ValueError(f'{path} 中 data 包含 NaN 或 Inf。')
    if config['nonnegative'] and torch.any(data < 0):
        raise ValueError(f'{path} 中 raw_polarity data 包含负值。')
    return data, metadata


def metadata_values_equal(left, right):
    if isinstance(left, torch.Tensor) and isinstance(right, torch.Tensor):
        return torch.equal(left, right)
    return type(left) is type(right) and left == right


def validate_preserved_fields(source_payload, output_payload, path):
    source_keys = set(source_payload) - {'data'}
    output_keys = set(output_payload) - {'data'}
    if output_keys != source_keys:
        raise ValueError(f'{path} 未完整保留源样本的非 data 字段。')
    for key in source_keys:
        if not metadata_values_equal(source_payload[key], output_payload[key]):
            raise ValueError(f'{path} 中字段 {key} 与源样本不一致。')


def threshold_scores(data, config):
    if config['score_transform'] == 'absolute':
        scores = data.abs()
    elif config['score_transform'] == 'identity':
        scores = data
    else:
        raise ValueError(f"未知的分数变换：{config['score_transform']}")

    if T == NATIVE_TIME_STEPS:
        return scores.contiguous()

    flat_scores = scores.flatten(start_dim=1)
    binned = torch.zeros((T, flat_scores.shape[1]), dtype=flat_scores.dtype)
    expanded_index = TIME_BIN_INDEX[:, None].expand_as(flat_scores)
    if hasattr(binned, 'scatter_reduce_'):
        # 最大值池化保证固定阈值下与原生时间轴脉冲的区间逻辑 OR 等价。
        binned.scatter_reduce_(0, expanded_index, flat_scores, reduce='amax', include_self=True)
    else:
        # 旧版 PyTorch 的兼容路径只循环 T 次，不改变区间定义。
        for bin_index in range(T):
            binned[bin_index] = flat_scores[TIME_BIN_INDEX == bin_index].amax(dim=0)
    return binned.reshape(T, *data.shape[1:]).contiguous()


def encode_threshold_spikes(data, config, threshold):
    return (threshold_scores(data, config) > threshold).contiguous()


def build_output_payload(source_payload, spikes):
    preserved = {}
    for key, value in source_payload.items():
        if key == 'data':
            continue
        preserved[key] = value.clone() if isinstance(value, torch.Tensor) else value
    return {'data': spikes, **preserved}


def validate_output_payload(source_payload, output_payload, path, channels):
    if not isinstance(output_payload, dict):
        raise TypeError(f'{path} 的内容必须是字典。')
    validate_metadata(output_payload, path)
    validate_preserved_fields(source_payload, output_payload, path)
    data = output_payload['data']
    expected_shape = (T, channels, 8, 16)
    if not isinstance(data, torch.Tensor):
        raise TypeError(f'{path} 中 data 必须是 Tensor。')
    if tuple(data.shape) != expected_shape or data.dtype != torch.bool:
        raise ValueError(
            f'{path} 中 data 为 shape={tuple(data.shape)}, dtype={data.dtype}；'
            f'预期 shape={expected_shape}, dtype=torch.bool。'
        )
    return data


def output_is_valid(path, source_payload, channels):
    if not path.is_file():
        return False
    try:
        validate_output_payload(source_payload, load_pt(path), path, channels)
        return True
    except (EOFError, KeyError, OSError, RuntimeError, TypeError, ValueError):
        return False

In [ ]:
source_files = {}
source_fingerprints = {}

for source_name, config in SOURCE_CONFIGS.items():
    if not config['root'].is_dir():
        raise FileNotFoundError(f"缺少数据目录：{config['root']}")
    if not config['manifest'].is_file():
        raise FileNotFoundError(f"缺少源数据清单：{config['manifest']}")

    source_fingerprints[source_name] = {
        'root': config['root'].relative_to(PROJECT_ROOT).as_posix(),
        'manifest': config['manifest'].relative_to(PROJECT_ROOT).as_posix(),
        'manifest_sha256': sha256sum(config['manifest']),
    }
    source_files[source_name] = {}
    for split_name in SPLIT_NAMES:
        files = sorted((config['root'] / split_name).glob('subject_*/*.pt'))
        if len(files) != EXPECTED_COUNTS[split_name]:
            raise RuntimeError(
                f'{source_name}/{split_name} 共有 {len(files)} 个样本，'
                f"预期 {EXPECTED_COUNTS[split_name]} 个。"
            )
        source_files[source_name][split_name] = files

print('源数据文件数量检查通过：')
for source_name in SOURCE_CONFIGS:
    print(source_name, {name: len(paths) for name, paths in source_files[source_name].items()})

In [ ]:
def estimate_threshold(source_name, config, paths):
    maximum = 0.0
    total_positions = 0
    with tqdm(paths, unit='trial', desc=f'{source_name} 阈值范围') as progress:
        for path in progress:
            payload = load_pt(path)
            data, metadata = validate_source_payload(payload, path, config)
            if metadata['repetition_id'] not in SPLIT_REPETITIONS['train']:
                raise ValueError(f'{path} 不属于训练 repetition。')
            scores = threshold_scores(data, config)
            maximum = max(maximum, float(scores.max()))
            total_positions += scores.numel()

    if not math.isfinite(maximum) or maximum <= 0:
        raise RuntimeError(f'{source_name} 的训练集 Threshold 最大值异常：{maximum}')

    histogram = torch.zeros(HISTOGRAM_BINS, dtype=torch.int64)
    with tqdm(paths, unit='trial', desc=f'{source_name} 阈值直方图') as progress:
        for path in progress:
            data, _ = validate_source_payload(load_pt(path), path, config)
            scores = threshold_scores(data, config)
            sample_histogram = torch.histc(
                scores,
                bins=HISTOGRAM_BINS,
                min=0.0,
                max=maximum,
            )
            histogram += sample_histogram.round().to(torch.int64)

    histogram_total = int(histogram.sum())
    if histogram_total != total_positions:
        raise RuntimeError(
            f'{source_name} 的直方图计数 {histogram_total} 与总位置数 {total_positions} 不一致。'
        )

    target_spikes = int(round(total_positions * TARGET_SPIKE_RATE))
    count_above = 0
    selected_bin = None
    for bin_index in range(HISTOGRAM_BINS - 1, -1, -1):
        bin_count = int(histogram[bin_index])
        if count_above + bin_count >= target_spikes:
            selected_bin = bin_index
            needed_from_bin = target_spikes - count_above
            fraction_above = needed_from_bin / bin_count if bin_count else 0.0
            bin_width = maximum / HISTOGRAM_BINS
            bin_upper = (bin_index + 1) * bin_width
            threshold = bin_upper - fraction_above * bin_width
            break
        count_above += bin_count

    if selected_bin is None:
        raise RuntimeError(f'{source_name} 无法从训练集直方图确定阈值。')

    return {
        'threshold': threshold,
        'target_spike_rate': float(TARGET_SPIKE_RATE),
        'quantile': 1.0 - float(TARGET_SPIKE_RATE),
        'score_transform': config['score_transform'],
        'training_positions': total_positions,
        'histogram_bins': HISTOGRAM_BINS,
        'histogram_minimum': 0.0,
        'histogram_maximum': maximum,
        'selected_bin': selected_bin,
        'estimated_target_spikes': target_spikes,
    }


threshold_results = {}
for source_name, config in SOURCE_CONFIGS.items():
    threshold_results[source_name] = estimate_threshold(
        source_name,
        config,
        source_files[source_name]['train'],
    )

print('训练集阈值估计完成：')
for source_name, result in threshold_results.items():
    print(f"{source_name}: threshold={result['threshold']:.9g}")

In [ ]:
def manifest_matches_existing(manifest):
    parameters = manifest.get('parameters', {})
    if parameters.get('time_steps') != T:
        return False
    if not math.isclose(
        float(parameters.get('target_spike_rate', -1)),
        float(TARGET_SPIKE_RATE),
        rel_tol=0.0,
        abs_tol=1e-15,
    ):
        return False
    if manifest.get('sources') != source_fingerprints:
        return False
    existing_thresholds = manifest.get('thresholds', {})
    for source_name, result in threshold_results.items():
        existing = existing_thresholds.get(source_name, {}).get('threshold')
        if existing is None or not math.isclose(
            float(existing), result['threshold'], rel_tol=1e-12, abs_tol=1e-15
        ):
            return False
    return True


existing_files = list(OUTPUT_ROOT.glob('raw_polarity/**/*.pt')) if OUTPUT_ROOT.is_dir() else []
existing_manifest = None
if OUTPUT_MANIFEST_PATH.is_file():
    existing_manifest = json.loads(OUTPUT_MANIFEST_PATH.read_text(encoding='utf-8'))
    if not manifest_matches_existing(existing_manifest) and not OVERWRITE:
        raise RuntimeError(
            '输出目录已有参数或源指纹不匹配的 manifest；请更换参数目录，'
            '或确认后将 OVERWRITE 设为 True。'
        )
elif existing_files and not OVERWRITE:
    raise RuntimeError(
        '输出目录已有 .pt 文件但没有可核对的 manifest；为避免混用编码参数，'
        '请确认后将 OVERWRITE 设为 True。'
    )

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
records = []
conversion_stats = Counter()
spike_statistics = {
    source_name: {
        split_name: {
            'spike_count': 0,
            'total_positions': 0,
            'channel_spike_counts': [0] * config['channels'],
        }
        for split_name in SPLIT_NAMES
    }
    for source_name, config in SOURCE_CONFIGS.items()
}
total_trials = len(SOURCE_CONFIGS) * sum(EXPECTED_COUNTS.values())

with tqdm(total=total_trials, unit='trial', desc='Threshold Encoding') as progress:
    for source_name, config in SOURCE_CONFIGS.items():
        threshold = threshold_results[source_name]['threshold']
        for split_name in SPLIT_NAMES:
            output_split_root = OUTPUT_ROOT / source_name / split_name
            for source_path in source_files[source_name][split_name]:
                source_payload = load_pt(source_path)
                data, metadata = validate_source_payload(source_payload, source_path, config)
                if metadata['repetition_id'] not in SPLIT_REPETITIONS[split_name]:
                    raise ValueError(
                        f"{source_path} 的 repetition_id={metadata['repetition_id']} "
                        f'不属于 {split_name}。'
                    )

                relative_path = source_path.relative_to(config['root'] / split_name)
                output_path = output_split_root / relative_path
                output_path.parent.mkdir(parents=True, exist_ok=True)
                expected_spikes = encode_threshold_spikes(data, config, threshold)

                reuse_existing = False
                if not OVERWRITE and output_is_valid(
                    output_path, source_payload, config['channels']
                ):
                    output_payload = load_pt(output_path)
                    reuse_existing = torch.equal(output_payload['data'], expected_spikes)

                if reuse_existing:
                    spikes = output_payload['data']
                    conversion_stats['reused'] += 1
                else:
                    spikes = expected_spikes
                    output_payload = build_output_payload(source_payload, spikes)
                    validate_output_payload(
                        source_payload, output_payload, output_path, config['channels']
                    )
                    save_pt_atomic(output_path, output_payload)
                    conversion_stats['written'] += 1

                stats = spike_statistics[source_name][split_name]
                stats['spike_count'] += int(spikes.sum())
                stats['total_positions'] += spikes.numel()
                channel_counts = spikes.sum(dim=(0, 2, 3), dtype=torch.int64)
                for channel_index, count in enumerate(channel_counts.tolist()):
                    stats['channel_spike_counts'][channel_index] += count

                records.append({
                    'source': source_name,
                    'source_path': source_path.relative_to(PROJECT_ROOT).as_posix(),
                    'output_path': output_path.relative_to(PROJECT_ROOT).as_posix(),
                    'split': split_name,
                    **metadata,
                    'data_shape': [T, config['channels'], 8, 16],
                    'data_dtype': 'torch.bool',
                })
                progress.update(1)
                progress.set_postfix(source=source_name, split=split_name)

for source_splits in spike_statistics.values():
    for stats in source_splits.values():
        stats['spike_rate'] = stats['spike_count'] / stats['total_positions']
        positions_per_channel = stats['total_positions'] // len(stats['channel_spike_counts'])
        stats['channel_spike_rates'] = [
            count / positions_per_channel for count in stats['channel_spike_counts']
        ]

print(f'编码完成：{dict(conversion_stats)}')
for source_name in SOURCE_CONFIGS:
    rates = {
        split_name: spike_statistics[source_name][split_name]['spike_rate']
        for split_name in SPLIT_NAMES
    }
    print(source_name, {name: f'{rate:.4%}' for name, rate in rates.items()})

In [ ]:
observed_counts = {}
verification_counts = Counter()

with tqdm(total=total_trials, unit='trial', desc='全量复核') as progress:
    for source_name, config in SOURCE_CONFIGS.items():
        observed_counts[source_name] = {}
        for split_name in SPLIT_NAMES:
            output_paths = sorted(
                (OUTPUT_ROOT / source_name / split_name).glob('subject_*/*.pt')
            )
            observed_counts[source_name][split_name] = len(output_paths)
            if len(output_paths) != EXPECTED_COUNTS[split_name]:
                raise RuntimeError(
                    f'{source_name}/{split_name} 输出 {len(output_paths)} 个样本，'
                    f"预期 {EXPECTED_COUNTS[split_name]} 个。"
                )

            for source_path in source_files[source_name][split_name]:
                relative_path = source_path.relative_to(config['root'] / split_name)
                output_path = OUTPUT_ROOT / source_name / split_name / relative_path
                source_payload = load_pt(source_path)
                data, _ = validate_source_payload(source_payload, source_path, config)
                output_payload = load_pt(output_path)
                output_data = validate_output_payload(
                    source_payload, output_payload, output_path, config['channels']
                )
                expected_data = encode_threshold_spikes(
                    data, config, threshold_results[source_name]['threshold']
                )
                if not torch.equal(output_data, expected_data):
                    raise ValueError(f'{output_path} 与当前 Threshold Encoding 参数不一致。')
                verification_counts['validated'] += 1
                progress.update(1)

for source_name in SOURCE_CONFIGS:
    training_rate = spike_statistics[source_name]['train']['spike_rate']
    if abs(training_rate - TARGET_SPIKE_RATE) > 0.01:
        raise RuntimeError(
            f'{source_name} 训练集实际发放率 {training_rate:.4%} 与目标'
            f' {TARGET_SPIKE_RATE:.4%} 相差超过 1 个百分点。'
        )

manifest = {
    'format_version': 1,
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'dataset': 'CapgMyo DB-a Threshold Encoding spike data',
    'output_root': OUTPUT_ROOT.relative_to(PROJECT_ROOT).as_posix(),
    'parameters': {
        'time_steps': T,
        'target_spike_rate': float(TARGET_SPIKE_RATE),
        'overwrite': OVERWRITE,
    },
    'sources': source_fingerprints,
    'encoding': {
        'raw_score': 'abs(X)',
        'raw_polarity_score': 'X_positive and X_negative, encoded independently',
        'temporal_binning': 'bin(t) = floor(t * T / 1000), non-overlapping maximum',
        'spike': 'S = temporal_bin_max(score) > threshold',
        'channel_count': 'preserved',
        'threshold_scope': 'one global scalar per source, estimated from train only',
        'output_dtype': 'torch.bool',
    },
    'thresholds': threshold_results,
    'split_counts': observed_counts,
    'spike_statistics': spike_statistics,
    'verification': {
        'validated_trials': verification_counts['validated'],
        'expected_trials': total_trials,
        'metadata_preserved': True,
        'binary_dtype_validated': True,
        'exact_encoding_validated': True,
    },
    'records': records,
}
write_json_atomic(OUTPUT_MANIFEST_PATH, manifest)

print('全量复核通过。')
print(f'输出文件数量：{observed_counts}')
print(f'manifest：{OUTPUT_MANIFEST_PATH}')
print('训练集实际发放率：')
for source_name in SOURCE_CONFIGS:
    stats = spike_statistics[source_name]['train']
    print(
        f"{source_name}: {stats['spike_rate']:.4%}, "
        f"threshold={threshold_results[source_name]['threshold']:.9g}"
    )